In [23]:
!python --version

Python 3.11.9


In [24]:
%matplotlib inline

In [25]:
import os

def define_auth():
    os.environ["HF_TOKEN"] = ""

In [26]:
import pandas as pd

def get_dataset(type="train"):
    splits = {'train': 'train_meta.csv', 'test': 'test_meta.csv'}
    df = pd.read_csv("hf://datasets/lasfk/EEG-fNIRS-based-Handwriting-Trajectory-Dataset/" + splits[type])
    return df

## EEG Extraction

In [27]:
import mne

def read_bdf(subject, session):
    path = f"datasets/raw/{subject}/EEG/{session}.bdf"
    raw = mne.io.read_raw_bdf(path, preload=True)
    return raw

In [28]:
import numpy as np

def clean_bdf(raw):
    empty_channels = [
    'Fpz',
    'Fp1',
    'Fp2',
    'AF3',
    'AF4',
    'AF7',
    'AF8',
    'F7',
    'F8',
    'FT7',
    'FT8',
    'T7',
    'T8',
    'TP7',
    'TP8',
    'Pz',
    'P3',
    'P4',
    'P5',
    'P6',
    'P7',
    'P8',
    'POz',
    'PO3',
    'PO4',
    'PO5',
    'PO6',
    'PO7',
    'PO8',
    'Oz',
    'O1',
    'O2',
    'ECG',
    'HEOR',
    'HEOL',
    'VEOU',
    'VEOL',
    ]
    raw.drop_channels(empty_channels)

def segmentation_eeg(raw, onset_sec, label, tmin, tmax):
    sfreq = raw.info["sfreq"]
    start_sample = int((onset_sec + tmin) * sfreq)
    end_sample = int((onset_sec + tmax) * sfreq)
    
    segment = raw.get_data(
        start=start_sample,
        stop=end_sample
    )

    expected_length = int((tmax - tmin) * sfreq)

    # Skip broken segment
    if segment.shape[1] != expected_length:
        return [], -1

    return segment, label

def normalize_channel(X):
    X_norm = np.zeros_like(X)

    for i in range(X.shape[0]):
    
        for ch in range(X.shape[1]):
    
            signal = X[i, ch]
    
            mean = signal.mean()
            std = signal.std()
    
            if std == 0:
                std = 1e-8
    
            X_norm[i, ch] = (signal - mean) / std
    return X_norm

def preprocessing_bdf(raw):
    clean_bdf(raw)

    # Buang sinyal listrik
    raw.notch_filter(50)

    # Band Pass 0.5Hz~40Hz, sinyal otak kecil, biasanya dalam range tersebut
    raw.filter(0.5, 40)

    # average reference
    raw.set_eeg_reference("average")

    return raw

## fNIRS Extraction

In [29]:
import pandas as pd

def read_fnirs(subject, session):
    path = f"datasets/raw/{subject}/fNIRS/{session}.csv"
    fnirs = pd.read_csv(path)
    return fnirs

In [30]:
import numpy as np
import pandas as pd

from scipy.signal import butter
from scipy.signal import sosfiltfilt
from tqdm import tqdm

def bandpass_filter(data, low, high, fs, order=4):
    nyquist = 0.5 * fs

    low = low / nyquist
    high = high / nyquist

    sos = butter(
        order,
        [low, high],
        btype="band",
        output="sos"
    )

    filtered = sosfiltfilt(
        sos,
        data
    )
    return filtered

def clean_fnirs(df, fs, fnirs_low, fnirs_high):

    fnirs_cols = [
        col for col in df.columns
        if col.startswith("fnirs_")
    ]

    # remove NaN
    df = df.dropna().reset_index(drop=True)

    # filtering
    for col in fnirs_cols:
        signal = df[col].values
        filtered = bandpass_filter(
            signal,
            fnirs_low,
            fnirs_high,
            fs
        )
        df[col] = filtered
    return df

def segmentation_fnirs(
    df,
    onset_sec,
    label,
    tmin,
    tmax,
    fs
):
    fnirs_cols = [
        col for col in df.columns
        if col.startswith("fnirs_")
    ]
    start = int((onset_sec + tmin) * fs)
    end = int((onset_sec + tmax) * fs)

    segment = df.iloc[start:end]
    expected_len = int((tmax - tmin) * fs)

    # skip broken segment
    if len(segment) != expected_len:
        return None, -1

    signal = segment[fnirs_cols].values.T

    return signal, label

def normalize_fnirs(X):
    X_norm = np.zeros_like(X)
    for i in range(X.shape[0]):

        for ch in range(X.shape[1]):

            signal = X[i, ch]

            mean = signal.mean()
            std = signal.std()

            if std == 0:
                std = 1e-8

            X_norm[i, ch] = (
                signal - mean
            ) / std

    return X_norm
    
def preprocessing_fnirs(df, fs, fnirs_low, fnirs_high):
    return clean_fnirs(df, fs, fnirs_low, fnirs_high)

## Extract Features

In [31]:
define_auth()
df_train = get_dataset(type="train")

print(df_train.head())

      trial_id subject  session  onset_sec  event_code  label
0  sub_01_1_00  sub_01        1     25.060         203      3
1  sub_01_1_01  sub_01        1     49.410         201      1
2  sub_01_1_02  sub_01        1     73.870         201      1
3  sub_01_1_03  sub_01        1     98.695         201      1
4  sub_01_1_04  sub_01        1    123.850         202      2


### Perbedaan trial onset_sec diff time

In [32]:
df_train["next_onset"] = (
    df_train["onset_sec"]
    .shift(-1)
)

df_train["diff"] = (
    df_train["next_onset"]
    -
    df_train["onset_sec"]
)

print(df_train["diff"].describe())

count    6442.000000
mean        0.137596
std       151.903929
min     -1345.378000
25%        24.852000
50%        24.858000
75%        24.864000
max       366.888000
Name: diff, dtype: float64


In [33]:
def clean_multimodal(
    X_eeg,
    X_fnirs,
    y
):

    clean_X_eeg = []
    clean_X_fnirs = []
    clean_y = []

    # 150 µV
    THRESHOLD = 150e-6

    for i in range(len(X_eeg)):

        eeg_signal = X_eeg[i]

        # cek apakah noisy
        if np.max(np.abs(eeg_signal)) < THRESHOLD:

            clean_X_eeg.append(
                X_eeg[i]
            )

            clean_X_fnirs.append(
                X_fnirs[i]
            )

            clean_y.append(
                y[i]
            )

    clean_X_eeg = np.array(
        clean_X_eeg
    )

    clean_X_fnirs = np.array(
        clean_X_fnirs
    )

    clean_y = np.array(
        clean_y
    )

    return (
        clean_X_eeg,
        clean_X_fnirs,
        clean_y
    )

### Extract Features EEG & fNIRS

In [34]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

mne.set_log_level("ERROR")

last_subject=""
last_session=0
raw=None
df_fnirs=None

EEG_TMIN = 0.0
EEG_TMAX = 4.0

# Dari explanatory data analysis
FNIRS_FS = 64

# 0.01 Hz ≤ f ≤ 0.2 Hz
# Kita hanya ambil perubahan sinyal yang:
# tidak terlalu lambat
# tidak terlalu cepat
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.2

# Dimulai + 2 Second = menunggu respons aliran darah mulai muncul.
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

X_eeg = []
X_fnirs = []
y = []
for index, row in  tqdm(df_train.iterrows(), total=len(df_train)):
    subject = row["subject"]
    session = row["session"]
    onset_sec = row["onset_sec"]
    label = row["label"]

    if last_subject != subject or last_session != session:
        # EEG
        raw = read_bdf(subject, session)
        raw = preprocessing_bdf(raw)

        # fNIRS
        df_fnirs = read_fnirs(subject, session)
        df_fnirs = preprocessing_fnirs(df_fnirs, FNIRS_FS, FNIRS_LOW, FNIRS_HIGH)
        
        last_subject = subject
        last_session = session

    # Segmentation BDF
    segment_eeg, label_eeg = segmentation_eeg(raw, onset_sec, label, EEG_TMIN, EEG_TMAX)
    
    # Segmentation fNIRS
    segment_fnirs, label_fnirs = segmentation_fnirs(
        df_fnirs,
        onset_sec,
        label,
        FNIRS_TMIN,
        FNIRS_TMAX,
        FNIRS_FS
    )
    if label_eeg == -1 or label_fnirs == -1:
        continue

    X_eeg.append(segment_eeg)
    X_fnirs.append(segment_fnirs)
    y.append(label)

X_eeg = np.array(X_eeg)
X_fnirs = np.array(X_fnirs)
y = np.array(y)

# Cleaning multimodal
X_eeg, X_fnirs, y = clean_multimodal(
    X_eeg,
    X_fnirs,
    y
)

X_eeg = np.nan_to_num(
    X_eeg,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_fnirs = np.nan_to_num(
    X_fnirs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalize EEG
X_eeg = normalize_channel(X_eeg)

# Normalize fNIRS
X_fnirs = normalize_fnirs(X_fnirs)

    


100%|██████████████████████████████████████████████████████████████████████████████| 6443/6443 [06:46<00:00, 15.83it/s]


## Split Dataset

In [35]:
from sklearn.model_selection import train_test_split

# First split:
# 80% train
# 20% temp

(
    X_eeg_train,
    X_eeg_temp,
    X_fnirs_train,
    X_fnirs_temp,
    y_train,
    y_temp
) = train_test_split(
    X_eeg,
    X_fnirs,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Second split:
# temp -> validation + test
# 10% validation
# 10% test

(
    X_eeg_val,
    X_eeg_test,
    X_fnirs_val,
    X_fnirs_test,
    y_val,
    y_test
) = train_test_split(
    X_eeg_temp,
    X_fnirs_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train EEG:", X_eeg_train.shape)
print("Validation EEG:", X_eeg_val.shape)
print("Test EEG:", X_eeg_test.shape)

print("Train fNIRS:", X_fnirs_train.shape)
print("Validation fNIRS:", X_fnirs_val.shape)
print("Test fNIRS:", X_fnirs_test.shape)

print("Train y:", y_train.shape)
print("Validation y:", y_val.shape)
print("Test y:", y_test.shape)

Train EEG: (4358, 27, 4000)
Validation EEG: (545, 27, 4000)
Test EEG: (545, 27, 4000)
Train fNIRS: (4358, 8, 384)
Validation fNIRS: (545, 8, 384)
Test fNIRS: (545, 8, 384)
Train y: (4358,)
Validation y: (545,)
Test y: (545,)


## Training Model

### Check Device

In [36]:
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(device)

True
NVIDIA GeForce RTX 4070
cuda


## Merge Training + Validation

In [37]:
import numpy as np

X_eeg_train_final = np.concatenate(
    [
        X_eeg_train,
        X_eeg_val
    ],
    axis=0
)

X_fnirs_train_final = np.concatenate(
    [
        X_fnirs_train,
        X_fnirs_val
    ],
    axis=0
)

y_train_final = np.concatenate(
    [
        y_train,
        y_val
    ],
    axis=0
)

print("Final EEG train shape:")
print(X_eeg_train_final.shape)

print("Final fNIRS train shape:")
print(X_fnirs_train_final.shape)

print("Final y train shape:")
print(y_train_final.shape)

Final EEG train shape:
(4903, 27, 4000)
Final fNIRS train shape:
(4903, 8, 384)
Final y train shape:
(4903,)


## Classifier

### Init Dataset Class

In [38]:
import torch
from torch.utils.data import Dataset

class MultiModalDataset(Dataset):
    def __init__(self, X_eeg, X_fnirs, y):
        self.X_eeg = torch.tensor(
            X_eeg,
            dtype=torch.float32
        )

        self.X_fnirs = torch.tensor(
            X_fnirs,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_eeg[idx],
            self.X_fnirs[idx],
            self.y[idx]
        )

### Init DataLoader

In [39]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

BATCH_SIZE = 32

train_dataset = MultiModalDataset(
    X_eeg_train_final,
    X_fnirs_train_final,
    y_train_final
)

test_dataset = MultiModalDataset(
    X_eeg_test,
    X_fnirs_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)



### Init MultiModalNet Class

In [40]:
import torch
import torch.nn as nn

class MultiModalNet(nn.Module):
    def __init__(self, eeg_channels, fnirs_channels, num_classes=4):

        super().__init__()

        # EEG Encoder
        self.eeg_encoder = nn.Sequential(
            nn.Conv1d(
                in_channels=eeg_channels,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                64,
                128,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        # fNIRS Encoder
        self.fnirs_encoder = nn.Sequential(
            nn.Conv1d(
                in_channels=fnirs_channels,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        # fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(
                128 + 64,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, eeg, fnirs):
        # EEG feature
        eeg_feat = self.eeg_encoder(eeg)

        # fNIRS feature
        fnirs_feat = self.fnirs_encoder(fnirs)

        # Fusion
        fused = torch.cat(
            [eeg_feat, fnirs_feat],
            dim=1
        )

        # Classification
        out = self.classifier(fused)

        return out

### Initialize Model, Criterion, Optimizer

In [41]:
model = MultiModalNet(
    eeg_channels=X_eeg.shape[1],
    fnirs_channels=X_fnirs.shape[1]
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

### Training Phase

In [42]:
from tqdm import tqdm

# =========================================================
# TRAINING
# =========================================================

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    # tqdm progress bar
    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=True
    )

    for eeg, fnirs, labels in train_bar:

        eeg = eeg.to(device)
        fnirs = fnirs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            eeg,
            fnirs
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            preds == labels
        ).sum().item()

        train_total += labels.size(0)

        train_acc = train_correct / train_total

        # update tqdm text
        train_bar.set_postfix({

            "loss": f"{loss.item():.4f}",

            "acc": f"{train_acc:.4f}"
        })

    epoch_loss = train_loss / len(train_loader)

    epoch_acc = train_correct / train_total

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}] "
        f"| Train Loss: {epoch_loss:.4f} "
        f"| Train Acc: {epoch_acc:.4f}"
    )

# =========================================================
# TEST EVALUATION
# =========================================================

model.eval()

test_correct = 0
test_total = 0

test_bar = tqdm(
    test_loader,
    desc="Testing",
    leave=True
)

with torch.no_grad():

    for eeg, fnirs, labels in test_bar:

        eeg = eeg.to(device)
        fnirs = fnirs.to(device)
        labels = labels.to(device)

        outputs = model(
            eeg,
            fnirs
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        test_correct += (
            preds == labels
        ).sum().item()

        test_total += labels.size(0)

        test_acc = test_correct / test_total

        test_bar.set_postfix({

            "acc": f"{test_acc:.4f}"
        })

test_acc = test_correct / test_total

print(f"\nTest Accuracy: {test_acc:.4f}")

Epoch 1/20: 100%|███████████████████████████████████████████| 154/154 [00:02<00:00, 67.94it/s, loss=1.3825, acc=0.2588]



Epoch [1/20] | Train Loss: 1.3891 | Train Acc: 0.2588


Epoch 2/20: 100%|██████████████████████████████████████████| 154/154 [00:01<00:00, 119.37it/s, loss=1.3108, acc=0.2870]



Epoch [2/20] | Train Loss: 1.3765 | Train Acc: 0.2870


Epoch 3/20: 100%|██████████████████████████████████████████| 154/154 [00:01<00:00, 119.94it/s, loss=1.3728, acc=0.2978]


...


Epoch [19/20] | Train Loss: 1.2857 | Train Acc: 0.3865


Epoch 20/20: 100%|█████████████████████████████████████████| 154/154 [00:01<00:00, 118.10it/s, loss=1.2817, acc=0.3834]



Epoch [20/20] | Train Loss: 1.2785 | Train Acc: 0.3834


Testing: 100%|████████████████████████████████████████████████████████████| 18/18 [00:00<00:00, 199.33it/s, acc=0.3028]


Test Accuracy: 0.3028


## Submission

In [43]:
import pandas as pd

df_test = get_dataset(type="test")
print(df_test.head())

      trial_id subject  session  onset_sec
0  sub_21_1_00  sub_21        1     24.495
1  sub_21_1_01  sub_21        1     49.347
2  sub_21_1_02  sub_21        1     74.205
3  sub_21_1_03  sub_21        1     99.075
4  sub_21_1_04  sub_21        1    123.933


In [44]:
import mne

mne.set_log_level("ERROR")
last_subject=""
last_session=0
raw=None
df_fnirs=None

EEG_TMIN = 0.0
EEG_TMAX = 4.0

# Dari explanatory data analysis
FNIRS_FS = 64

# 0.01 Hz ≤ f ≤ 0.2 Hz
# Kita hanya ambil perubahan sinyal yang:
# tidak terlalu lambat
# tidak terlalu cepat
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.2

# Dimulai + 2 Second = menunggu respons aliran darah mulai muncul.
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

X_test_eeg = []
X_test_fnirs = []
trial_ids = []

for _, row in tqdm(
    df_test.iterrows(),
    total=len(df_test)
):
    subject = row["subject"]
    session = row["session"]
    onset_sec = row["onset_sec"]
    trial_id = row["trial_id"]

    if last_subject != subject or last_session != session:
        # EEG
        raw = read_bdf(subject, session)
        raw = preprocessing_bdf(raw)

        # fNIRS
        df_fnirs = read_fnirs(subject, session)
        df_fnirs = preprocessing_fnirs(df_fnirs, FNIRS_FS, FNIRS_LOW, FNIRS_HIGH)
        
        last_subject = subject
        last_session = session
        
    # Segmentation BDF
    segment_eeg, label_eeg = segmentation_eeg(raw, onset_sec, label, EEG_TMIN, EEG_TMAX)
    
    # Segmentation fNIRS
    segment_fnirs, label_fnirs = segmentation_fnirs(
        df_fnirs,
        onset_sec,
        label,
        FNIRS_TMIN,
        FNIRS_TMAX,
        FNIRS_FS
    )
    if label_eeg == -1 or label_fnirs == -1:
        continue

    X_test_eeg.append(
        segment_eeg
    )

    X_test_fnirs.append(
        segment_fnirs
    )

    trial_ids.append(
        trial_id
    )

X_test_eeg = np.array(X_test_eeg)
X_test_fnirs = np.array(X_test_fnirs)

X_test_eeg = np.nan_to_num(
    X_test_eeg,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_test_fnirs = np.nan_to_num(
    X_test_fnirs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalize EEG
X_test_eeg = normalize_channel(X_test_eeg)

# Normalize fNIRS
X_test_fnirs = normalize_fnirs(X_test_fnirs)

print("Test EEG shape:")
print(X_test_eeg.shape)

print("Test fNIRS shape:")
print(X_test_fnirs.shape)


100%|██████████████████████████████████████████████████████████████████████████████| 3238/3238 [03:30<00:00, 15.40it/s]


Test EEG shape:
(3205, 27, 4000)
Test fNIRS shape:
(3205, 8, 384)


In [45]:
import torch

X_test_eeg_tensor = torch.tensor(
    X_test_eeg,
    dtype=torch.float32
)

X_test_fnirs_tensor = torch.tensor(
    X_test_fnirs,
    dtype=torch.float32
)

In [47]:
from tqdm import tqdm
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.eval()

predictions = []

with torch.no_grad():

    predict_bar = tqdm(
        range(len(X_test_eeg_tensor)),
        desc="Predicting"
    )

    for i in predict_bar:

        eeg = (
            X_test_eeg_tensor[i]
            .unsqueeze(0)
            .to(device)
        )

        fnirs = (
            X_test_fnirs_tensor[i]
            .unsqueeze(0)
            .to(device)
        )

        outputs = model(
            eeg,
            fnirs
        )

        pred = torch.argmax(
            outputs,
            dim=1
        ).item()

        predictions.append(pred)

        predict_bar.set_postfix({

            "last_pred": pred
        })

Predicting: 100%|████████████████████████████████████████████████████| 3205/3205 [00:04<00:00, 699.74it/s, last_pred=0]


In [48]:
submission = pd.DataFrame({
    "trial_id": trial_ids,
    "label": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved!")

submission.csv saved!
